# Grok-infra-t4x2-smoke

Remote GPU smoke test for **Kaggle T4 x2**.

Goals:
1. Confirm CUDA + 2x T4 are visible
2. Run a multi-GPU tensor op
3. Train a tiny model for a few steps
4. Write `/kaggle/working/result.json` as success artifact


In [ ]:
import json, os, platform, time, sys
from pathlib import Path

print("python", sys.version)
print("platform", platform.platform())
print("cwd", os.getcwd())
print("kaggle input", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "n/a")


In [ ]:
import torch

assert torch.cuda.is_available(), "CUDA not available — accelerator not attached"
n = torch.cuda.device_count()
print("torch", torch.__version__)
print("cuda", torch.version.cuda)
print("device_count", n)
for i in range(n):
    print(f"  gpu[{i}]", torch.cuda.get_device_name(i),
          "mem_gb", round(torch.cuda.get_device_properties(i).total_memory / 1e9, 2))

# Prefer 2x T4; accept 1x if quota only grants single
assert n >= 1, "Need at least 1 GPU"
names = [torch.cuda.get_device_name(i) for i in range(n)]
print("devices:", names)


In [ ]:
import torch

# Multi-GPU bandwidth-ish smoke: matmul on each device + all_reduce via sum on CPU
torch.manual_seed(42)
results = []
for i in range(torch.cuda.device_count()):
    device = torch.device(f"cuda:{i}")
    a = torch.randn(4096, 4096, device=device)
    b = torch.randn(4096, 4096, device=device)
    torch.cuda.synchronize(device)
    t0 = time.time()
    c = a @ b
    torch.cuda.synchronize(device)
    dt = time.time() - t0
    s = float(c.sum().item())
    results.append({"device": i, "name": torch.cuda.get_device_name(i), "matmul_s": dt, "sum": s})
    print(f"device {i}: matmul {dt:.4f}s sum={s:.4f}")

assert all(r["matmul_s"] > 0 for r in results)
gpu_results = results


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Tiny multi-GPU training on a learnable synthetic task (not pure noise)
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(128, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )
    def forward(self, x):
        return self.net(x)

device = torch.device("cuda:0")
model = TinyNet().to(device)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print("using DataParallel on", torch.cuda.device_count(), "GPUs")

opt = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

def make_batch(bs=256):
    x = torch.randn(bs, 128, device=device)
    # label depends on first 10 dims so model can learn quickly
    y = (x[:, :10].sum(dim=1) > 0).long() % 10
    y = (x[:, :10].argmax(dim=1))
    return x, y

losses = []
t0 = time.time()
for step in range(80):
    x, y = make_batch()
    opt.zero_grad(set_to_none=True)
    logits = model(x)
    loss = loss_fn(logits, y)
    loss.backward()
    opt.step()
    losses.append(float(loss.item()))
    if step % 10 == 0 or step == 79:
        print(f"step {step:02d} loss={loss.item():.4f}")

train_s = time.time() - t0
print(f"train_time_s={train_s:.3f} final_loss={losses[-1]:.4f}")
# allow small variance but require meaningful drop
assert losses[-1] < losses[0] * 0.95, f"loss did not drop enough: {losses[0]} -> {losses[-1]}"
train_meta = {"steps": len(losses), "loss_start": losses[0], "loss_end": losses[-1], "train_s": train_s}


In [ ]:
import json
from pathlib import Path

out = {
    "ok": True,
    "notebook": "Grok-infra-t4x2-smoke",
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "device_count": torch.cuda.device_count(),
    "devices": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "gpu_matmul": gpu_results,
    "train": train_meta,
}
Path("/kaggle/working").mkdir(parents=True, exist_ok=True)
path = Path("/kaggle/working/result.json")
path.write_text(json.dumps(out, indent=2))
print("wrote", path)
print(json.dumps(out, indent=2))
print("SMOKE_OK")
